# Kontroly společného modelu Random Forest

**Autor:** Pavlo Romanovyč Prytula

Pomocné výstupy ke společnému modelu [`rf_model.ipynb`](rf_model.ipynb) (appendix prezentace).
Čísla citovaná v `rf_model.ipynb` pocházejí odsud. Běh trvá asi 2 minuty.

In [1]:
%run rf_model.ipynb

import numpy as np
from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

strata = df["hotel"] + "_" + df[TARGET].astype(str)

Bookings: 118558 | attributes: 22


Train: 94847 | test: 23711 (20.0%)
Feature vectors shared by train and test: 0


,train cancel rate,test cancel rate,test bookings
hotel,,,
City Hotel,0.419,0.419,15778
Resort Hotel,0.280,0.280,7933


ROC-AUC train: 0.956
ROC-AUC test:  0.893
Accuracy test: 0.830
City Hotel: 15778 test bookings, ROC-AUC 0.903
Resort Hotel: 7933 test bookings, ROC-AUC 0.860


## A. Proč skupinové rozdělení: náhodné rozdělení a duplicitní rezervace

Stejný model natrénovaný na obyčejném náhodném rozdělení (80/20, stratifikace `hotel × is_canceled`).
Testovací rezervace dělím na ty, které mají identickou kopii (stejný vektor 22 atributů) v trénovacích
datech, a ostatní.

In [2]:
idx_train, idx_test = train_test_split(df.index, test_size=0.2, random_state=SEED, stratify=strata)
random_model = clone(pipeline).fit(df.loc[idx_train, FEATURES], df.loc[idx_train, TARGET])
proba = pd.Series(random_model.predict_proba(df.loc[idx_test, FEATURES])[:, 1], index=idx_test)
y_rand = df.loc[idx_test, TARGET]

seen = pd.Series(np.isin(group_ids[idx_test], group_ids[idx_train]), index=idx_test)
print(f"ROC-AUC test (random split): {roc_auc_score(y_rand, proba):.3f}")
print(f"Test bookings with an identical copy in train: {seen.mean():.1%} ({seen.sum()})")

rows = []
for name, part in [("copy in train", seen), ("no copy in train", ~seen)]:
    rows.append({"part": name, "bookings": part.sum(), "cancel rate": y_rand[part].mean(),
                 "ROC-AUC": roc_auc_score(y_rand[part], proba[part])})
display(pd.DataFrame(rows).set_index("part").round(3))

hotel_test = df.loc[idx_test, "hotel"]
display(seen.groupby(hotel_test).mean().rename("share with copy in train").round(3))

ROC-AUC test (random split): 0.914
Test bookings with an identical copy in train: 37.2% (8823)


,bookings,cancel rate,ROC-AUC
part,,,
copy in train,8823,0.536,0.981
no copy in train,14889,0.276,0.831


hotel
City Hotel      0.434
Resort Hotel    0.249
Name: share with copy in train, dtype: float64

Při náhodném rozdělení má 37 % testovacích rezervací (City Hotel 43 %, Resort Hotel 25 %) identickou
kopii v trénovacích datech. Na nich model dosahuje ROC-AUC 0,981, na ostatních jen 0,831 – model kopie
„pozná“ a celkové ROC-AUC 0,914 je nadsazené. Navíc se to týká hotelů nestejně, což by zkreslilo jejich
srovnání. Proto společný model používá skupinové rozdělení.

## B. Stabilita kvality: skupinová křížová validace na trénovacích datech

Pět foldů `StratifiedGroupKFold` pouze uvnitř trénovacích dat (testovací data se nepoužijí).

In [3]:
cv_train = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
aucs = []
for tr, va in cv_train.split(X_train, strata[X_train.index], group_ids[X_train.index]):
    fold_model = clone(pipeline).fit(X_train.iloc[tr], y_train.iloc[tr])
    aucs.append(roc_auc_score(y_train.iloc[va], fold_model.predict_proba(X_train.iloc[va])[:, 1]))
print("ROC-AUC per fold:", np.round(aucs, 3))
print(f"Mean ± std: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

ROC-AUC per fold: [0.883 0.888 0.882 0.872 0.886]
Mean ± std: 0.882 ± 0.006


ROC-AUC v jednotlivých foldech je 0,872–0,888 (0,882 ± 0,006). Testovací fold (0,893) je mírně příznivý,
realistická kvalita na nových rezervacích je ≈ 0,88–0,89.

## C. Technické kontroly modelu

In [4]:
pre = pipeline.named_steps["preprocess"]
encoded_test = pre.transform(X_test)
print("Encoded columns in the order of FEATURES:", list(pre.get_feature_names_out()) == FEATURES)
print("Unknown categories in test (code -1):",
      {c: int((encoded_test[:, i] == -1).sum()) for i, c in enumerate(CATEGORICAL) if (encoded_test[:, i] == -1).any()})

sizes = pd.Series(group_ids).value_counts()
mixed = df.groupby(group_ids)[TARGET].nunique() > 1
print("Groups:", len(sizes), "| groups with more than 1 booking:", (sizes > 1).sum(), "| largest group:", sizes.max())
print("Groups with both outcomes:", mixed.sum(), "| bookings in them:", int(sizes[mixed[mixed].index].sum()))

Encoded columns in the order of FEATURES: True
Unknown categories in test (code -1): {'agent': 17, 'company': 30}
Groups: 81133 | groups with more than 1 booking: 9583 | largest group: 180
Groups with both outcomes: 677 | bookings in them: 5211


Pořadí zakódovaných sloupců odpovídá `FEATURES`; neznámé kategorie se týkají jen několika desítek
rezervací (`agent`, `company`). 677 skupin shodných rezervací (5 211 rezervací) má oba výsledky –
takové případy žádný model nerozliší.

## D. Anomálie `previous_cancellations`

In [5]:
prev = df[df["previous_cancellations"] == 1]
print("Bookings with previous_cancellations = 1:", len(prev), f"| cancel rate: {prev[TARGET].mean():.1%}")
print(f"Non Refund: {(prev['deposit_type'] == 'Non Refund').mean():.0%} | repeated guest: {prev['is_repeated_guest'].mean():.0%}")
display(prev["market_segment"].value_counts(normalize=True).round(2).head(4))

Bookings with previous_cancellations = 1: 6044 | cancel rate: 94.5%
Non Refund: 58% | repeated guest: 11%


market_segment
Groups           0.54
Offline TA/TO    0.27
Online TA        0.09
Corporate        0.07
Name: proportion, dtype: float64

Hodnota 1 má 6 044 rezervací, z nich 94,5 % je zrušeno; 54 % jsou `Groups`, 58 % má nevratnou zálohu
a jen 11 % je označeno jako opakovaný host. Atribut je známý při rezervaci (není to leakage),
ale spíše zachycuje artefakt skupinových rezervací než skutečnou historii hosta. Ponecháváme ho
a v úlohách PFI, ALE a SHAP tuto výhradu uvádíme.

## E. Atributy vznikající po rezervaci: doba do zrušení

Pokud atribut přibývá s časem po rezervaci, zrušené rezervace (které „žily“ kratší dobu) ho mají méně –
model by se pak učil z informace, která v okamžiku rezervace ještě není. Datum zrušení je jen v surových
datech (`reservation_status_date`), proto tato kontrola pracuje s `data/raw/hotel_bookings.csv`.
Pro zrušené rezervace s `lead_time` 90–240 dní počítám, jakou část doby mezi rezervací a příjezdem
rezervace existovala, a podíl rezervací se speciálním požadavkem / se změnou.

In [6]:
raw = pd.read_csv("../../../data/raw/hotel_bookings.csv")
arrival = pd.to_datetime(
    raw["arrival_date_year"].astype(str) + "-" + raw["arrival_date_month"] + "-"
    + raw["arrival_date_day_of_month"].astype(str), format="%Y-%B-%d"
)
booked = arrival - pd.to_timedelta(raw["lead_time"], unit="D")
raw["days_alive"] = (pd.to_datetime(raw["reservation_status_date"]) - booked).dt.days

period = raw["lead_time"].between(90, 240)
canceled = raw[period & (raw["is_canceled"] == 1)].copy()
canceled["canceled_after"] = pd.cut(
    canceled["days_alive"] / canceled["lead_time"], [-0.01, 0.1, 0.5, 0.9, 1.01],
    labels=["first 10 %", "10-50 %", "50-90 %", "last 10 %"],
)

def share_table(data):
    return data.groupby("canceled_after", observed=True).agg(
        bookings=("days_alive", "size"),
        special_request=("total_of_special_requests", lambda s: (s > 0).mean()),
        booking_change=("booking_changes", lambda s: (s > 0).mean()),
    ).round(3)

print("Canceled bookings, all deposit types:")
display(share_table(canceled))
print("Canceled bookings, No Deposit only:")
display(share_table(canceled[canceled["deposit_type"] == "No Deposit"]))

kept = raw[period & (raw["is_canceled"] == 0) & (raw["deposit_type"] == "No Deposit")]
print(f"Not canceled, No Deposit: special request {(kept['total_of_special_requests'] > 0).mean():.3f}, "
      f"booking change {(kept['booking_changes'] > 0).mean():.3f}")
print("Share of adr == 0 by target:", df.groupby(TARGET)["adr"].apply(lambda s: (s == 0).mean()).round(3).to_dict())

Canceled bookings, all deposit types:


,bookings,special_request,booking_change
canceled_after,,,
first 10 %,5604,0.202,0.044
10-50 %,4978,0.225,0.061
50-90 %,4166,0.281,0.106
last 10 %,1749,0.361,0.125


Canceled bookings, No Deposit only:


,bookings,special_request,booking_change
canceled_after,,,
first 10 %,3590,0.316,0.069
10-50 %,2715,0.412,0.108
50-90 %,2933,0.399,0.138
last 10 %,1641,0.383,0.130


Not canceled, No Deposit: special request 0.546, booking change 0.227
Share of adr == 0 by target: {0: 0.013, 1: 0.004}


- `booking_changes` jasně přibývá s dobou existence rezervace (4 % → 13 % u všech zrušených),
  vyloučení atributu je tedy oprávněné.
- `total_of_special_requests`: u všech zrušených rezervací podíl také roste (20 % → 36 %), ale z velké části
  proto, že brzy zrušené rezervace mají často nevratnou zálohu a ty požadavky téměř nemají. U rezervací
  bez zálohy je nárůst malý (32 % → ≈ 40 %) a i pozdě zrušené rezervace mají požadavek mnohem méně často
  než nezrušené. Atribut ponecháváme, s výhradou.
- `adr == 0` je vzácné v obou třídách, `adr` neukazuje známky leakage.